# Causal Survival Analysis & Heterogeneous Treatment Effects with PyTorch
This notebook demonstrates how to utilize real survival data from the **GBSG2** (German Breast Cancer Study Group 2) dataset, preprocess clinical covariates safely, and implement a **Causal Survival Neural Network (Two-Head Architecture)** in PyTorch to estimate causal risks and treatment effects.

In [ ]:
# 1. Install scikit-survival for accessing benchmark survival datasets
!pip install -q scikit-survival

# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pandas as pd
from sksurv.datasets import load_gbsg2

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Select computation device (GPU/CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### 2. Load and Preprocess GBSG2 Dataset
We load the GBSG2 breast cancer dataset, encode categorical features, extract treatment indicator (`horTh_yes`), survival time, and event status, and perform safe numerical standardization.

In [ ]:
# Load GBSG2 dataset from scikit-survival
data_x, data_y = load_gbsg2()

# One-hot encode categorical features
data_x_encoded = pd.get_dummies(data_x, drop_first=True)

# Extract treatment indicator (horTh: hormone therapy yes/no)
if 'horTh_yes' in data_x_encoded.columns:
    T = data_x_encoded['horTh_yes'].astype(int).values
    X_features = data_x_encoded.drop(columns=['horTh_yes']).values
else:
    # Fallback treatment assignment if column name differs
    T = np.random.binomial(1, 0.5, size=len(data_x))
    X_features = data_x_encoded.values

# Convert X_features explicitly to a float numpy array
X_features = np.array(X_features, dtype=float)

# Extract survival event status and time from structured array data_y
event = np.array([row[0] for row in data_y], dtype=float)
time = np.array([row[1] for row in data_y], dtype=float)

# Compute mean and standard deviation safely using numpy functions
X_mean = np.mean(X_features, axis=0)
X_std = np.std(X_features, axis=0) + 1e-8
# Prevent division by zero for columns with constant values
X_std[X_std == 1e-8] = 1.0

# Normalize features
X_scaled = (X_features - X_mean) / X_std

print(f"Dataset Feature Shape: {X_scaled.shape}")
print(f"Treatment Distribution -> Control (0): {sum(T==0)}, Treated (1): {sum(T==1)}")
print(f"Overall Event Rate: {event.mean():.4f}")

# Preview dataset as a Pandas DataFrame
preview_df = pd.DataFrame(X_scaled, columns=[f"Feature_{i+1}" for i in range(X_scaled.shape[1])])
preview_df['Treatment'] = T
preview_df['Event'] = event
preview_df['Time'] = time
display(preview_df.head())

### 3. Prepare PyTorch Tensors and DataLoader
Wrap preprocessed arrays into PyTorch Tensors and create mini-batches using `DataLoader`.

In [ ]:
# Convert numpy arrays to PyTorch tensors
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
T_tensor = torch.tensor(T, dtype=torch.float32).unsqueeze(1)
time_tensor = torch.tensor(time, dtype=torch.float32).unsqueeze(1)
event_tensor = torch.tensor(event, dtype=torch.float32).unsqueeze(1)

# Create dataset and dataloader
dataset = TensorDataset(X_tensor, T_tensor, time_tensor, event_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

### 4. Design Causal Survival Neural Network Architecture
Implement a Two-Head architecture inspired by TARNet to estimate potential risks for control and treatment groups.

In [ ]:
class CausalSurvivalNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, rep_dim=32):
        super(CausalSurvivalNet, self).__init__()
        
        # Shared representation layers
        self.representation = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, rep_dim),
            nn.ReLU()
        )
        # Outcome Head for Control group (T = 0)
        self.head_0 = nn.Sequential(
            nn.Linear(rep_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        # Outcome head for Treated group (T = 1)
        self.head_1 = nn.Sequential(
            nn.Linear(rep_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        phi = self.representation(x)
        risk_0 = self.head_0(phi)
        risk_1 = self.head_1(phi)
        return risk_0, risk_1

# Initialize model and move to device
model = CausalSurvivalNet(input_dim=X_scaled.shape[1]).to(device)
print(model)

### 5. Training Loop
Optimize the network using Adam optimizer and factual risk matching.

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
mse_loss = nn.MSELoss()

epochs = 60
model.train()

for epoch in range(epochs):
    total_loss = 0.0
    for batch_x, batch_t, batch_time, batch_event in dataloader:
        batch_x = batch_x.to(device)
        batch_t = batch_t.to(device)
        batch_time = batch_time.to(device)
        batch_event = batch_event.to(device)
        
        risk_0, risk_1 = model(batch_x)
        
        # Select factual risk based on actual treatment assignment
        factual_risk = batch_t * risk_1 + (1.0 - batch_t) * risk_0
        
        # Supervised loss weighted by event status
        normalized_time = batch_time / (batch_time.max() + 1e-8)
        loss = mse_loss(factual_risk * batch_event, normalized_time)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch_x.size(0)
        
        if (epoch + 1) % 15 == 0:
            pass
        
    if (epoch + 1) % 15 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {total_loss / len(dataset):.4f}")

### 6. Evaluate Treatment Effects
Compute individual risk differences between treated and control groups.

In [ ]:
model.eval()
with torch.no_grad():
    X_tensor_dev = X_tensor.to(device)
    r0, r1 = model(X_tensor_dev)
    risk_diff = (r1 - r0).cpu().numpy().flatten()

print(f"Average Risk Difference (Treatment Effect): {risk_diff.mean():.4f}")
print("Sample Individual Risk Differences:", risk_diff[:5])